<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session10_fine_tuning_with_lora_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۰: فاین‌تیون مدل‌های زبانی با LoRA و QLoRA

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session10_fine_tuning_with_lora.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** هر تمرین یک یا چند بار مدل را آموزش می‌دهد (هر بار حدود ۵ تا ۱۰ دقیقه روی T4).

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install -U transformers trl peft bitsandbytes datasets huggingface_hub
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import json
import os
import re
import time

import pandas as pd
import torch
from datasets import Dataset
from llm_utils import download_data
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


def load_jsonl(name):
    with open(download_data(name), encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_data, test_data, real_data = (load_jsonl(f"orders_{s}.jsonl") for s in ("train", "test", "real"))
new_templates = test_data[150:]

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM_SHORT = ("Extract the order from the Persian customer message as JSON with keys "
                "product_id, quantity, color, city.")
FIELDS = ["product_id", "quantity", "color", "city"]
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def conversations(data):
    return [[{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}] for d in data]


def generate(model, convs, batch_size=16, max_new_tokens=80):
    tokenizer.padding_side = "left"
    outputs = []
    for i in range(0, len(convs), batch_size):
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs[i:i + batch_size]]
        batch = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(out[:, batch.input_ids.shape[1]:], skip_special_tokens=True)
    return outputs


def parse(output):
    match = re.search(r"\{.*\}", output, re.S)
    try:
        return json.loads(match.group()) if match else None
    except json.JSONDecodeError:
        return None


def same(a, b):
    norm = lambda x: None if x in (None, "", "null") else str(x).replace("\u200c", " ").strip()
    return norm(a) == norm(b)


def score(outputs, data):
    rows = []
    for out, item in zip(outputs, data):
        pred = parse(out) or {}
        row = {"message": item["message"], "output": out.strip(), "valid_json": bool(pred)}
        row.update({f: bool(pred) and same(pred.get(f), item["order"][f]) for f in FIELDS})
        row["all_fields"] = all(row[f] for f in FIELDS)
        rows.append(row)
    return pd.DataFrame(rows)


def to_dataset(data):
    return Dataset.from_list([
        {"prompt": [{"role": "system", "content": SYSTEM_SHORT}, {"role": "user", "content": d["message"]}],
         "completion": [{"role": "assistant", "content": json.dumps(d["order"], ensure_ascii=False)}]}
        for d in data])


def train(data, r=16, output_dir="order-extractor-lora", epochs=2):
    """Train a QLoRA adapter and return (model, training seconds)."""
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
    tokenizer.padding_side = "right"
    trainer = SFTTrainer(
        model=model, train_dataset=to_dataset(data), processing_class=tokenizer,
        peft_config=LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules="all-linear",
                               task_type="CAUSAL_LM"),
        args=SFTConfig(output_dir=output_dir, num_train_epochs=epochs, per_device_train_batch_size=8,
                       gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                       warmup_steps=0.05, logging_steps=20, max_length=256, fp16=True, bf16=False,
                       save_strategy="no", report_to="none"))
    start = time.time()
    trainer.train()
    seconds = time.time() - start
    trainer.model.save_pretrained(output_dir)
    trainer.model.eval()
    return trainer.model, seconds


def free(*objects):
    for o in objects:
        del o
    torch.cuda.empty_cache()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

آموزش را با `r=4` و `r=64` تکرار کنید (و `lora_alpha` را دو برابر `r` بگذارید). دقت روی الگوهای جدید، زمان آموزش و حجم آداپتور چه تغییری می‌کند؟

</div>

In [ ]:
rows = []
for r in [4, 16, 64]:
    model, seconds = train(train_data, r=r, output_dir=f"lora-r{r}")
    acc = score(generate(model, conversations(new_templates)), new_templates).all_fields.mean()
    size_mb = os.path.getsize(f"lora-r{r}/adapter_model.safetensors") / 1e6
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    rows.append({"r": r, "trainable_params": trainable, "adapter_MB": round(size_mb, 1),
                 "train_minutes": round(seconds / 60, 1), "new_templates_accuracy": round(acc, 3)})
    del model
    torch.cuda.empty_cache()
pd.DataFrame(rows)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

حجم آداپتور و تعداد پارامترها تقریباً خطی با `r` زیاد می‌شوند، ولی زمان آموزش تغییر چندانی نمی‌کند، چون بیشتر محاسبه در مدل پایه انجام می‌شود، نه در ماتریس‌های کوچک LoRA.

برای کاری به این محدودی، `r=4` هم معمولاً ظرفیت کافی دارد و افزایش `r` به ۶۴ لزوماً دقت را بالا نمی‌برد. رتبهٔ بالاتر برای کارهای پیچیده‌تر یا داده‌های متنوع‌تر لازم می‌شود. با ۵۰ پیام آزمون، اختلاف یکی دو درصدی بین `r`ها معنادار نیست.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

منحنی یادگیری بسازید: مدل را با ۱۰۰، ۳۰۰ و ۱۲۰۰ نمونهٔ آموزشی آموزش دهید و دقت کامل را روی آزمون رسم کنید. چند نمونه برای این کار کافی است؟

</div>

In [ ]:
import matplotlib.pyplot as plt

sizes, accuracies = [100, 300, 1200], []
for n in sizes:
    model, _ = train(train_data[:n], output_dir=f"lora-n{n}")
    accuracies.append(score(generate(model, conversations(new_templates)), new_templates).all_fields.mean())
    del model
    torch.cuda.empty_cache()

plt.plot(sizes, accuracies, marker="o")
plt.xscale("log")
plt.xlabel("training examples")
plt.ylabel("all fields correct (new templates)")
plt.ylim(0, 1)
plt.show()
print(dict(zip(sizes, [round(a, 3) for a in accuracies])))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با ۲ دورهٔ آموزش، ۱۰۰ نمونه فقط حدود ۱۲ گام به‌روزرسانی است. پس بخشی از ضعف دادهٔ کم، از **گام‌های کم** می‌آید، نه فقط از نمونه‌های کم. برای مقایسهٔ منصفانه‌تر می‌توان تعداد کل گام‌ها را ثابت نگه داشت، یعنی برای داده‌های کوچک‌تر تعداد دوره را بیشتر کرد.

یک نکتهٔ مهم دیگر: دادهٔ مصنوعی ما از ۸ الگو ساخته شده است. بعد از اینکه مدل همهٔ الگوها و کالاها را چند بار دید، نمونهٔ بیشتر از **همان الگوها** کمک زیادی نمی‌کند. برای تعمیم بهتر، **تنوع** داده مهم‌تر از **تعداد** آن است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

مدل فاین‌تیون‌شده را روی `real_data` ارزیابی کنید و خطاها را دسته‌بندی کنید. کدام ویژگی پیام‌های واقعی در دادهٔ مصنوعی نبوده است؟

</div>

In [ ]:
model, _ = train(train_data)
real_scores = score(generate(model, conversations(real_data)), real_data)
print(f"real messages, all fields correct: {real_scores.all_fields.mean():.2f}")
pd.set_option("display.max_colwidth", 80)
real_scores.query("not all_fields")[["message", "output", *FIELDS]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ویژگی‌هایی از پیام‌های واقعی که در دادهٔ مصنوعی نبوده‌اند:

| ویژگی | نمونه |
|---|---|
| نوشتار محاوره‌ای | «واسه آمل»، «برا شرکتمون»، «لپتاپ» بدون نیم‌فاصله |
| عدد چسبیده | «۲تا»، «دوتا» |
| نام کالا با حروف فارسی یا ترتیب متفاوت | «پوکو ایکس ۷»، «TUF ایسوس» |
| ساختار جمله متفاوت | «من از کرج سفارش میدم: ...»، شهر در ابتدای پیام |
| اطلاعات اضافه | «برای خونه جدیدمون»، «برای کافی‌نت» |

برای بهبود، سه راه وجود دارد: افزودن این سبک‌ها به اسکریپت تولید داده، تولید داده با یک مدل بزرگ‌تر به جای الگو، و جمع‌آوری پیام‌های واقعی و برچسب‌زدن آن‌ها. راه سوم بهترین است. حتی صد پیام واقعی معمولاً بیش از هزار پیام مصنوعی اثر دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

فرض کنید فروشگاه کالای جدیدی با شناسهٔ `P1007` اضافه می‌کند. مدل فاین‌تیون‌شده با پیامی دربارهٔ این کالا چه می‌کند؟ دو راه برای پشتیبانی از کالاهای جدید پیشنهاد دهید و مزایا و معایب هر کدام را بنویسید.

</div>

In [ ]:
new_product_messages = [
    {"message": "سلام، یه گوشی موتورولا G85 آبی برای تهران می‌خوام",
     "order": {"product_id": "P1007", "quantity": 1, "color": "آبی", "city": "تهران"}},
    {"message": "دو تا موتو G85 مشکی بفرستید شیراز",
     "order": {"product_id": "P1007", "quantity": 2, "color": "مشکی", "city": "شیراز"}},
]
for out, item in zip(generate(model, conversations(new_product_messages)), new_product_messages):
    print(item["message"], "\n  ->", out.strip())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل شناسهٔ `P1007` را هرگز ندیده است، پس یکی از شناسه‌های آشنا را برمی‌گرداند، احتمالاً گوشی‌ای با رنگ یا قیمت نزدیک. این خطرناک‌ترین نوع خطاست: خروجی معتبر به نظر می‌رسد ولی کالای اشتباه سفارش داده می‌شود.

**دو راه‌حل:**

| راه | مزیت | عیب |
|---|---|---|
| ۱. آموزش دوباره با دادهٔ کالای جدید | پرامپت کوتاه می‌ماند | برای هر تغییر فهرست کالا باید آموزش داد؛ کند و پرهزینه |
| ۲. تقسیم کار: مدل **نام کالا** را استخراج کند و کد آن را با فهرست کالا تطبیق دهد | کالای جدید فقط با به‌روزرسانی فهرست اضافه می‌شود؛ «پیدا نشد» قابل تشخیص است | مرحلهٔ تطبیق نام باید خوب ساخته شود، مثلاً با جستجوی معنایی جلسهٔ ۶ |

راه دوم همان درس جلسهٔ ۸ است: **دانش متغیر، یعنی فهرست کالاها، را بیرون از وزن‌های مدل نگه دارید** و مدل را برای کاری آموزش دهید که ثابت است، یعنی فهم پیام و قالب خروجی. نمونهٔ سادهٔ مرحلهٔ تطبیق:

</div>

In [ ]:
import difflib

catalog = {"P1001": ["سامسونگ A55", "گلکسی A55"], "P1005": ["آیفون 16", "iPhone 16"],
           "P1007": ["موتورولا G85", "موتو G85"]}          # the new product is just a new row


def match_product(name, cutoff=0.6):
    aliases = {alias: pid for pid, names in catalog.items() for alias in names}
    best = difflib.get_close_matches(name, list(aliases), n=1, cutoff=cutoff)
    return aliases[best[0]] if best else None              # None means: ask the customer


for name in ["موتورولا G85", "موتو جی ۸۵", "گلکسی A55", "شیائومی ۱۵"]:
    print(f"{name:15} -> {match_product(name)}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`difflib` فقط شباهت نوشتاری را می‌سنجد و «موتو جی ۸۵» را پیدا نمی‌کند. برای تطبیق واقعی، از جستجوی معنایی با امبدینگ (جلسهٔ ۶) یا ترکیب آن با شباهت نوشتاری استفاده کنید. مهم این است که خروجی `None` امکان‌پذیر است و سیستم می‌تواند از مشتری بپرسد منظورش کدام کالاست. همین ایده را در جلسهٔ ۱۵ کامل‌تر می‌سازیم.

</div>